# PySpark Data Transformation Pipeline
Update the configuration in Cell 2, then run the notebook from top to bottom. The pipeline reads a CSV, applies reusable cleansing and type-standardization rules, validates output quality, and writes Parquet.
from __future__ import annotations


In [8]:
import logging
import os
from pathlib import Path

from common.spark import build_spark_session

logging.basicConfig(level=logging.INFO)


MINIO_BUCKET = os.environ.get("MINIO_BUCKET", "data-bucket")
# Define directory paths
BRONZE_DIR = f"s3a://{MINIO_BUCKET}/bronze"
FIXTURES_DIR = Path.cwd() / "tests" / "fixtures"
SILVER_DIR = f"s3a://{MINIO_BUCKET}/silver"
GOLD_DIR = f"s3a://{MINIO_BUCKET}/gold"


## 1. Initialize PySpark Session


In [9]:
spark = build_spark_session()
print(f"Spark {spark.version} is ready with MinIO integration.")
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 100)


Spark 3.5.5 is ready with MinIO integration.


In [10]:
from common.utils import (
    build_create_table_sql,
    load_dataframe,
    required_columns,
    store_df_to_table,
    validate_required_columns,
)


## 2. Load Source DataFrames


In [11]:
df = load_dataframe(spark, f"{BRONZE_DIR}/input.csv", input_format="csv")
df.show()
print(f"Rows: {df.count()}")
df.printSchema()


+-----------+-------------+------+------+----------+
|customer_id|customer_name|region|amount|order_date|
+-----------+-------------+------+------+----------+
|        101|        Alice| North| 250.0|2024-01-05|
|        102|          Bob| South| 180.5|2024-01-08|
|        103|      Charlie|  East|320.75|2024-01-12|
|        104|        David|  West| 210.0|2024-01-15|
|        105|          Eve| North|410.25|2024-01-18|
|        106|        Frank| South| 195.4|2024-01-22|
|        107|        Grace|  East| 275.8|2024-01-25|
|        108|        Henry|  West| 330.1|2024-01-29|
|        109|          Ivy| North| 260.6|2024-02-02|
|        110|         Jack| South| 420.9|2024-02-05|
+-----------+-------------+------+------+----------+

Rows: 10
root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- order_date: date (nullable = true)



## Cast dataframe columns, include metadata and descriptions
    spark.sql(f"DESCRIBE TABLE EXTENDED {table_name}").show(truncate=False)
Define all column metadata in a dictionary, then generate SQL dynamically.

In [12]:



SILVER_TABLE_NAME = "silver.customer_orders"

# Column metadata: "source"/"type"/"is_required" drive the catalog table, the rest feeds the data dictionary.
SILVER_COLUMN_CONFIG = {
    "customer_id": {
        "source": "customer_id",
        "type": "INT",
        "is_required": True,
        "description": "Unique identifier for the customer who placed the order.",
        "valid_range": ">= 1, non-null",
    },
    "customer_name": {
        "source": "customer_name",
        "type": "STRING",
        "is_required": True,
        "description": "Customer's display name.",
    },
    "region": {
        "source": "region",
        "type": "STRING",
        "is_required": True,
        "description": "Sales region the order belongs to.",
    },
    "amount": {
        "source": "amount",
        "type": "DECIMAL(18,2)",
        "is_required": True,
        "description": "Order amount, cleaned and cast from the bronze source.",
        "unit": "USD (assumed \u2014 confirm currency with source system)",
        "valid_range": ">= 0",
    },
    "order_date": {
        "source": "order_date",
        "type": "DATE",
        "is_required": True,
        "description": "Date the order was placed.",
        "valid_range": "not in the future",
    },
}

SILVER_TABLE_METADATA = {
    "comment": "Cleaned, validated, and deduplicated customer order data (bronze to silver).",
    "grain": "One row per customer_id.",
    "primary_key": ["customer_id"],
    "freshness": (
        "Updated daily by the `customer-bronze-to-gold` Prefect deployment "
        "(cron 0 2 * * * UTC), ahead of the gold stage."
    ),
    "caveats": [
        "Despite covering 'orders', deduplication is keyed only on customer_id "
        "(clean_dataframe with source_key_columns=['customer_id']), so there is at most "
        "one row per customer, not one row per order.",
        "region enum values have not been confirmed against the source system; treat as free text.",
    ],
}

REQUIRED_COLUMNS = required_columns(SILVER_COLUMN_CONFIG)
validate_required_columns(df, REQUIRED_COLUMNS)


create_table_sql = build_create_table_sql(
    table_name=SILVER_TABLE_NAME,
    column_config=SILVER_COLUMN_CONFIG,
    location=f"{SILVER_DIR}/customer_orders",
    table_format="PARQUET",
    partition_columns=["region"],
    table_comment=SILVER_TABLE_METADATA["comment"],
)

store_df_to_table(spark,
    df=df,
    table_name=SILVER_TABLE_NAME,
    column_config=SILVER_COLUMN_CONFIG,
    create_table_sql=create_table_sql,
    write_mode="overwrite"
)

INFO:common.utils:All required columns are present: ['amount', 'customer_id', 'customer_name', 'order_date', 'region']


In [13]:
# Verify the inserted data.
spark.sql(f"""
    SELECT *
    FROM {SILVER_TABLE_NAME}
""")
# spark.sql(create_t

customer_id,customer_name,amount,order_date,region
101,Alice,250.00,2024-01-05,North
105,Eve,410.25,2024-01-18,North
109,Ivy,260.60,2024-02-02,North
102,Bob,180.50,2024-01-08,South
106,Frank,195.40,2024-01-22,South
110,Jack,420.90,2024-02-05,South
103,Charlie,320.75,2024-01-12,East
107,Grace,275.80,2024-01-25,East
104,David,210.00,2024-01-15,West
108,Henry,330.10,2024-01-29,West


In [14]:
spark.sql(
    f"DESCRIBE EXTENDED {SILVER_TABLE_NAME}"
).show(truncate=False)


+----------------------------+----------------------------------------------------------------------------+--------------------------------------------------------+
|col_name                    |data_type                                                                   |comment                                                 |
+----------------------------+----------------------------------------------------------------------------+--------------------------------------------------------+
|customer_id                 |int                                                                         |Unique identifier for the customer who placed the order.|
|customer_name               |string                                                                      |Customer's display name.                                |
|amount                      |decimal(18,2)                                                               |Order amount, cleaned and cast from the bronze source.  |
|order_dat

In [15]:
spark._activeSession.stop()